# Notebook 01: Pelatihan & Tuning Mandiri Model Deteksi Wajah YOLOv8 (Epoch 0)

Notebook ini khusus menjalankan **Fokus 1**, yaitu membangun, melatih, dan men-tuning model detektor wajah **YOLOv8 Polosan (`arsitektur_yolov8_wajah.yaml`)** dari **Epoch 0** menggunakan **GPU NVIDIA GeForce RTX 4060 Laptop GPU** hingga menghasilkan bobot optimal:
- `storage/app/ai/weights/hasil_training_yolov8_wajah.pt`

Setelah file bobot `hasil_training_yolov8_wajah.pt` terbentuk dan mencapai skor `mAP` yang baik, proses dilanjutkan ke **Notebook 02 (`02_training_dataset_wajah_aktor/Notebook_02_Training_Dataset_Wajah_Aktor.ipynb`)**.

In [ ]:
# Mengatur path absolut root proyek agar seluruh modul terbaca dengan benar dari Jupyter Notebook lokal
import sys
from pathlib import Path

FOLDER_NOTEBOOK_INI = Path.cwd().resolve()
if FOLDER_NOTEBOOK_INI.name == "01_training_model_yolo":
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI.parent.parent
elif FOLDER_NOTEBOOK_INI.name == "training_python":
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI.parent
else:
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI

for path_modul in [
    FOLDER_ROOT_PROYEK,
    FOLDER_ROOT_PROYEK / "training_python",
    FOLDER_ROOT_PROYEK / "training_python" / "01_training_model_yolo",
    FOLDER_ROOT_PROYEK / "python",
]:
    if str(path_modul) not in sys.path:
        sys.path.insert(0, str(path_modul))

import konfigurasi_training as konfig
from persiapan_dataset_yolo import siapkan_dan_split_dataset_yolo
from training_dan_tuning_model_yolo import (
    muat_arsitektur_yolov8_polos,
    latih_model_yolov8_dari_nol,
    jalankan_tuning_hyperparameter_yolo,
    evaluasi_model_yolo_pada_split_test,
)

# Memverifikasi penguncian GPU NVIDIA GeForce RTX 4060 Laptop GPU (GPU 1 pada Task Manager / cuda:0 pada PyTorch)
indeks_cuda, objek_device_gpu, info_perangkat_gpu = konfig.pilih_perangkat_gpu_rtx_wajib(wajib_cuda=True)
model_yolo_uji = muat_arsitektur_yolov8_polos()

print("Root Proyek          :", FOLDER_ROOT_PROYEK)
print("Perangkat GPU Aktif  :", info_perangkat_gpu["nama_gpu"], f"({info_perangkat_gpu['vram_gb']} GB VRAM)")
print("Indeks CUDA          :", info_perangkat_gpu["device_pytorch"], "-", info_perangkat_gpu["catatan_task_manager"])
print("Arsitektur YOLOv8    :", konfig.FILE_ARSITEKTUR_YOLOV8_YAML)
print("Jumlah Parameter YOLO:", sum(p.numel() for p in model_yolo_uji.model.parameters()))

## Tahap 01: Persiapan, Validasi Label, & Pembagian Dataset Deteksi Wajah (`train / val / test`)
Mengambil gambar wajah manusia beserta label `.txt` dari `python/datasets/dataset_yolo_wajah/raw_wajah_manusia/` serta gambar negatif tanpa wajah (hewan/kera, benda, pemandangan) dari `raw_negatif_bukan_wajah/`, lalu membagi otomatis ke `split_yolo/` (`80% train`, `10% val`, `10% test`).

In [ ]:
# Menyiapkan dan membagi dataset deteksi wajah YOLOv8 beserta pembuatan file data_yolo_wajah.yaml
ringkasan_persiapan_dataset_yolo = siapkan_dan_split_dataset_yolo(
    rasio_train=konfig.RASIO_SPLIT_TRAIN_YOLO,
    rasio_val=konfig.RASIO_SPLIT_VAL_YOLO,
    rasio_test=konfig.RASIO_SPLIT_TEST_YOLO,
)

## Tahap 02: Pelatihan & Tuning Model YOLOv8 Polosan dari Epoch 0 di GPU RTX 4060
Melatih `arsitektur_yolov8_wajah.yaml` dari bobot awal acak (`pretrained=False`, `amp=False`) dengan parameter tuning optimizer, *Cosine Learning Rate*, bobot *Loss* (`box`, `cls`, `dfl`), serta *Early Stopping* (`patience`), lalu menyimpan hasil terbaik ke `storage/app/ai/weights/hasil_training_yolov8_wajah.pt`.

In [ ]:
# Melatih model YOLOv8 deteksi wajah dari Epoch 0 menggunakan GPU RTX 4060
data_hasil_training_yolo = latih_model_yolov8_dari_nol(
    jumlah_epoch=konfig.JUMLAH_EPOCH_TRAINING_YOLO,
    batas_patience=konfig.BATAS_PATIENCE_EARLY_STOPPING_YOLO,
    ukuran_gambar=konfig.UKURAN_GAMBAR_INPUT_YOLO,
    ukuran_batch=konfig.UKURAN_BATCH_TRAINING_YOLO,
)

### (Opsional) Eksperimen Perbandingan Hyperparameter Tuning YOLOv8
Menjalankan perbandingan beberapa skenario parameter (misal `AdamW` vs `SGD`) dari `DAFTAR_KANDIDAT_TUNING_YOLO` dan memilih bobot dengan skor `mAP50-95` tertinggi.

In [ ]:
# Menjalankan eksperimen perbandingan kandidat tuning hyperparameter YOLOv8 bila diperlukan
# laporan_tuning_yolo = jalankan_tuning_hyperparameter_yolo(jumlah_epoch_per_skenario=30, batas_patience=10)

## Tahap 03: Evaluasi Model YOLOv8 Hasil Training Mandiri pada Split Test
Menguji bobot `hasil_training_yolov8_wajah.pt` pada data uji (`split test`) untuk memperoleh skor `Precision`, `Recall`, `mAP50`, dan `mAP50-95`.

In [ ]:
# Mengevaluasi akurasi deteksi kotak wajah YOLOv8 pada split test
data_evaluasi_test_yolo = evaluasi_model_yolo_pada_split_test()
print(data_evaluasi_test_yolo)